# Black-Litterman Bayesian Portfolio Allocation

**Implementation of the Black-Litterman Asset Allocation Model (He & Litterman, 1999; Idzorek, 2005)**
- Market Equilibrium Prior Returns via Reverse Optimization (CAPM Implied $\boldsymbol{\Pi}$)
- Incorporating Absolute and Relative Quantitative Views with Confidence Scaling ($\Omega$ Matrix)
- Posterior Combined Return Vector $\mathbb{E}[R]$ and Posterior Covariance $\boldsymbol{\Sigma}_{\text{post}}$
- Smooth portfolio tilting without extreme Markowitz corner solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from black_litterman_engine import BlackLittermanModel

## 1. Market Equilibrium Setup (Reverse Optimization)

In [ ]:
assets = ["Apple (AAPL)", "Microsoft (MSFT)", "Amazon (AMZN)", "Alphabet (GOOGL)", "JPMorgan (JPM)", "ExxonMobil (XOM)"]
w_mkt = [0.25, 0.23, 0.16, 0.15, 0.11, 0.10]
vols = np.array([0.24, 0.22, 0.28, 0.26, 0.20, 0.22])
corr = np.full((6, 6), 0.45)
np.fill_diagonal(corr, 1.0)
cov = np.diag(vols) @ corr @ np.diag(vols)

bl = BlackLittermanModel(assets, cov, w_mkt, risk_aversion=3.0, tau=0.05)
pd.DataFrame({"Asset": assets, "Market Cap Weight": w_mkt, "Implied Equilibrium Return Pi": bl.pi})

## 2. Setting Absolute & Relative Investor Views

In [ ]:
# View 1: AAPL will return 9.0% (Absolute view, 70% confidence)
# View 2: MSFT will outperform AMZN by +3.5% (Relative view, 85% confidence)
P = [
    [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, -1.0, 0.0, 0.0, 0.0]
]
Q = [0.090, 0.035]
bl.set_views(P, Q, confidences=[0.70, 0.85])

post = bl.compute_posterior()
w_constrained = bl.compute_constrained_weights(post["posterior_returns"], allow_short=False)

## 3. Allocation Comparison & Tilting

In [ ]:
x = np.arange(len(assets))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, w_mkt, width, label='Benchmark Market Weight', color='steelblue')
ax.bar(x + width/2, w_constrained, width, label='Black-Litterman Optimal Weight', color='darkorange')
ax.set_ylabel('Portfolio Allocation Weight')
ax.set_title('Asset Allocation: Market Benchmark vs. Black-Litterman Tilt')
ax.set_xticks(x)
ax.set_xticklabels([a.split()[0] for a in assets])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()